In [0]:
pip install openpyxl

In [0]:
#Load Config and Setup Enviorment Variables
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
 
# print(f"env_code: {lz_key}")  # This won't be redacted
# print(f"env_name: {env_name}")  # This won't be redacted
 
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
# print(f"KeyVault_name: {KeyVault_name}")
 
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
 
# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"
  
# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
  

In [0]:
# Requires openpyxl — add to cell 1 if not installed: %pip install openpyxl

# ── OUTPUT CONFIG ──────────────────────────────────────────────────────────────
LOCAL_PATH = "/tmp/gold_data_sheet.xlsx"        # driver-local, openpyxl writes here
DBFS_PATH  = "dbfs:/tmp/gold_data_sheet.xlsx"   # copy here to make downloadable

# ── TAB NAMES ─────────────────────────────────────────────────────────────────
TAB_NAMES = {
    "paymentPending":                "PP-CaseData",
    "appealSubmitted":               "AS-CaseData",
    "awaitingRespondentEvidence(a)": "AREa-CaseData",
    "awaitingRespondentEvidence(b)": "AREb-CaseData",
    "caseUnderReview":               "CUR-CaseData",
    "reasonsForAppealSubmitted":     "RFAS-CaseData",
    "listing":                       "LST-CaseData",
    "prepareForHearing":             "PFH-CaseData",
    "decision":                      "DEC-CaseData",
    "decided(a)":                    "DECa-CaseData",
    "decided(b)":                    "DECb-CaseData",
    "ftpaSubmitted(a)":              "FTSa-CaseData",
    "ftpaSubmitted(b)":              "FTSb-CaseData",
    "ftpaDecided":                   "FTPD-CaseData",
    "ended":                         "END-CaseData",
    "remitted":                      "REM-CaseData",
}

ALL_STATES = [
    "paymentPending",
    "appealSubmitted",
    "awaitingRespondentEvidence(a)",
    "awaitingRespondentEvidence(b)",
    "caseUnderReview",
    "reasonsForAppealSubmitted",
    "listing",
    "prepareForHearing",
    "decision",
    "decided(a)",
    "decided(b)",
    "ftpaSubmitted(a)",
    "ftpaSubmitted(b)",
    "ftpaDecided",
    "ended",
    "remitted"
]

import json as _json
from openpyxl import Workbook

def _to_excel_val(v):
    """Convert a pandas cell value to something openpyxl can write.
    Dicts/lists (nested Spark structs/arrays) are serialised to a JSON string."""
    if v is None:
        return None
    if isinstance(v, (bool, int, float, str)):
        return v
    if str(v) in ("nan", "NaT", "None"):
        return None
    if isinstance(v, (dict, list)):
        try:
            return _json.dumps(v, default=str)
        except Exception:
            return str(v)
    return str(v)

wb = Workbook()
wb.remove(wb.active)  # remove default empty sheet

# ── Loop Each State and get Latest Gold ───────────────────────────────────────
for state_under_test in ALL_STATES:
    print(f"\nProcessing: {state_under_test}")

    bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
    silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
    gold_path   = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"

    # Get Latest Json Folder
    json_location        = dbutils.fs.ls(f"{gold_path}/")[-1]
    latest_json_location = json_location.name
    dbutils.fs.ls(f"{gold_path}/{latest_json_location}")

    # Set Paths
    try:
        json_path      = f"{gold_path}/{latest_json_location}/JSON/"
        M1_silver_path = f"{silver_path}/silver_appealcase_detail"
        M1_bronze_path = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
        M2_silver_path = f"{silver_path}/silver_caseapplicant_detail"
        M3_silver_path = f"{silver_path}/silver_status_detail"
        C_path         = f"{silver_path}/silver_appealcategory_detail"
        bhc_path       = f"{bronze_path}/bronze_hearing_centres"
        bat_path       = f"{bronze_path}/bronze_appealtype"
        docsr_path     = f"{bronze_path}/bronze_documentsreceived"
    except Exception as e:
        print(f"Error during path setup: {str(e)}")

    # Create and Load Dataframes
    json_data = spark.read.format("json").load(json_path)
    M1_silver = spark.read.format("delta").load(M1_silver_path)
    M1_bronze = spark.read.format("delta").load(M1_bronze_path)
    M2_silver = spark.read.format("delta").load(M2_silver_path)
    M3_silver = spark.read.format("delta").load(M3_silver_path)
    C         = spark.read.format("delta").load(C_path)
    bhc       = spark.read.format("delta").load(bhc_path)
    bat       = spark.read.format("delta").load(bat_path)
    docsr     = spark.read.format("delta").load(docsr_path)

    from pyspark.sql.functions import (
        col, when, lit, array, struct, collect_list,
        max as spark_max, date_format, row_number, expr,
        size, udf, coalesce, concat_ws, concat, trim, year, split, datediff,
        collect_set, current_timestamp, transform, first, array_contains
    )

    # Patch missing witness interpreter fields for later-chain states
    if state_under_test in ("prepareForHearing", "decision", "decided(a)", "decided(b)", "ended", "remitted"):
        from pyspark.sql.types import StructType, StructField, StringType
        schema = StructType([
            StructField("appealReferenceNumber",              StringType(), True),
            StructField("witness1InterpreterSignLanguage",    StringType(), True),
            StructField("witness2InterpreterSignLanguage",    StringType(), True),
            StructField("witness3InterpreterSignLanguage",    StringType(), True),
            StructField("witness4InterpreterSignLanguage",    StringType(), True),
            StructField("witness5InterpreterSignLanguage",    StringType(), True),
            StructField("witness6InterpreterSignLanguage",    StringType(), True),
            StructField("witness7InterpreterSignLanguage",    StringType(), True),
            StructField("witness8InterpreterSignLanguage",    StringType(), True),
            StructField("witness9InterpreterSignLanguage",    StringType(), True),
            StructField("witness10InterpreterSignLanguage",   StringType(), True),
            StructField("witness1InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness2InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness3InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness4InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness5InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness6InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness7InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness8InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness9InterpreterSpokenLanguage",  StringType(), True),
            StructField("witness10InterpreterSpokenLanguage", StringType(), True),
        ])
        missing_fields_json = spark.read.schema(schema).json(json_path)
        json_data = json_data.join(
            missing_fields_json,
            json_data["appealReferenceNumber"] == missing_fields_json["appealReferenceNumber"],
            "left"
        ).drop(missing_fields_json["appealReferenceNumber"])

    # ── Write gold dataframe to Excel tab ─────────────────────────────────────
    tab_name = TAB_NAMES.get(state_under_test, state_under_test[:31])
    print(f"  Converting to pandas for tab: {tab_name}")
    pdf = json_data.toPandas()

    ws = wb.create_sheet(title=tab_name)

    # Column headers at row 5, starting from column B (index 2)
    for col_idx, col_name in enumerate(pdf.columns, start=2):
        ws.cell(row=5, column=col_idx, value=col_name)

    # Data rows from row 6 onward — _to_excel_val handles nested dicts/lists
    for row_idx, row_data in enumerate(pdf.itertuples(index=False), start=6):
        for col_idx, value in enumerate(row_data, start=2):
            ws.cell(row=row_idx, column=col_idx, value=_to_excel_val(value))

    print(f"  {len(pdf)} rows, {len(pdf.columns)} columns written to '{tab_name}'")

# ── Save: write to driver /tmp first, then copy to DBFS ───────────────────────
# /dbfs/ FUSE mount does not support openpyxl's file operations (errno 95)
wb.save(LOCAL_PATH)
dbutils.fs.cp(f"file:{LOCAL_PATH}", DBFS_PATH)
print(f"\nWorkbook saved to DBFS: {DBFS_PATH}")
print("Download via the Databricks UI: Data > DBFS > tmp > gold_data_sheet.xlsx")